In [1]:
##### Evaluate RF models 
# calculates model performance metrics 

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

In [2]:
##### SET-UP

# Get the current working directory
cd = Path.cwd().parent 

# import reference data 
ref_data = pd.read_csv(f"{cd}/Data/subnational_intensities.csv")

# import model predictions (fold, split, and prediction columns already included)
capital_rf = pd.read_parquet(f'{cd}/Results/RF_models/capital_rf_spatial_CV/predictions.parquet')
labor_rf = pd.read_parquet(f'{cd}/Results/RF_models/labor_rf_spatial_CV/predictions.parquet')

capital_qrf = pd.read_parquet(f'{cd}/Results/RF_models/capital_qrf_spatial_CV/predictions.parquet')
labor_qrf = pd.read_parquet(f'{cd}/Results/RF_models/labor_qrf_spatial_CV/predictions.parquet')

In [3]:
##### PREP DATA

# Add original intensity data (sub-national and country) for evaluation
capital_rf = capital_rf.merge(ref_data, on=['PROJ_ID', 'country_ID'], how='left')
labor_rf = labor_rf.merge(ref_data, on=['PROJ_ID', 'country_ID'], how='left')
capital_qrf = capital_qrf.merge(ref_data, on=['PROJ_ID', 'country_ID'], how='left')
labor_qrf = labor_qrf.merge(ref_data, on=['PROJ_ID', 'country_ID'], how='left')

# Estimate actual predicted intensity (not relative)
capital_rf['predicted_log_capital_intensity_USD_per_million_tonne'] = capital_rf['prediction'] + capital_rf['log_country_capital_intensity_USD_per_million_tonne']
capital_rf['predicted_capital_intensity_USD_per_million_tonne'] = np.expm1(capital_rf['predicted_log_capital_intensity_USD_per_million_tonne'])
capital_qrf['q10_predicted_log_capital_intensity_USD_per_million_tonne'] = capital_qrf['q10'] + capital_qrf['log_country_capital_intensity_USD_per_million_tonne']
capital_qrf['q90_predicted_log_capital_intensity_USD_per_million_tonne'] = capital_qrf['q90'] + capital_qrf['log_country_capital_intensity_USD_per_million_tonne']

labor_rf['predicted_log_labor_intensity_jobs_per_million_tonne'] = labor_rf['prediction'] + labor_rf['log_country_labor_intensity_jobs_per_million_tonne']
labor_rf['predicted_labor_intensity_jobs_per_million_tonne'] = np.expm1(labor_rf['predicted_log_labor_intensity_jobs_per_million_tonne'])
labor_qrf['q10_predicted_log_labor_intensity_jobs_per_million_tonne'] = labor_qrf['q10'] + labor_qrf['log_country_labor_intensity_jobs_per_million_tonne']
labor_qrf['q90_predicted_log_labor_intensity_jobs_per_million_tonne'] = labor_qrf['q90'] + labor_qrf['log_country_labor_intensity_jobs_per_million_tonne']

In [4]:
##### DEFINE FUNCTIONS

# function to compute key performance metrics per fold 
def compute_metrics_by_fold(df, target_col, pred_col):
    rows = []
    for fold, g in df.groupby("fold"):
        y_true = g[target_col]
        y_pred = g[pred_col]
        resid  = y_true - y_pred

        rows.append({
            "fold":      fold,
            "n":         len(g),
            "R2":        r2_score(y_true, y_pred),
            "RMSE":      np.sqrt(mean_squared_error(y_true, y_pred)),
            "MAE":       mean_absolute_error(y_true, y_pred),
            "Bias":      resid.mean(),
            "Pearson_r": y_true.corr(y_pred),
        })

    metrics_df = pd.DataFrame(rows).sort_values("fold").reset_index(drop=True)

    # summary rows across folds (averaged, not pooled)
    metric_cols = ["R2", "RMSE", "MAE", "Bias", "Pearson_r"]
    mean_row = {"fold": "mean", "n": metrics_df["n"].sum()}
    var_row  = {"fold": "variance", "n": np.nan}
    for col in metric_cols:
        mean_row[col] = metrics_df[col].mean()
        var_row[col]  = metrics_df[col].var()  

    summary = pd.DataFrame([mean_row, var_row])
    return pd.concat([metrics_df, summary], ignore_index=True)

# function to evaluate prediction intervals from QRF, by fold
def compute_interval_metrics_by_fold(df, target_col, q10col, q90col):
    rows = []
    for fold, g in df.groupby("fold"):
        y_true = g[target_col]
        low    = g[q10col]
        high   = g[q90col]
        covered   = (y_true >= low) & (y_true <= high)

        rows.append({
            "fold":             fold,
            "n":                len(g),
            "coverage":         covered.mean(),
            "nominal_coverage": 0.8,
            "mean_width":       (high - low).mean(),
        })

    metrics_df = pd.DataFrame(rows).sort_values("fold").reset_index(drop=True)

    # summary rows across folds (averaged, not pooled)
    metric_cols = ["coverage", "mean_width"]
    mean_row = {"fold": "mean", "n": metrics_df["n"].sum(), "nominal_coverage": 0.8}
    var_row  = {"fold": "variance", "n": np.nan, "nominal_coverage": np.nan}
    for col in metric_cols:
        mean_row[col] = metrics_df[col].mean()
        var_row[col]  = metrics_df[col].var()

    summary = pd.DataFrame([mean_row, var_row])
    return pd.concat([metrics_df, summary], ignore_index=True)

In [5]:
##### CAPITAL MODEL METRICS

### Prep data ---------------------------------------------------------------------------

# set target column
target_col = "rtv_log_capital_intensity_USD_per_million_tonne" 

# set output directory
out_dir = Path(f'{cd}/Results/RF_models/capital_rf_spatial_CV')
out_dir_qrf = Path(f'{cd}/Results/RF_models/capital_qrf_spatial_CV')

# split into test and train
test_df  = capital_rf[capital_rf["split"] == "test"]
train_df = capital_rf[capital_rf["split"] == "train"]
test_df_qrf = capital_qrf[capital_qrf["split"] == "test"]

### Calculate metrics ----------------------------------------------------------------------

## RELATIVE INTENSITIES 
# training data
train_metrics = compute_metrics_by_fold(train_df, target_col, 'prediction')
# test data
test_metrics = compute_metrics_by_fold(test_df, target_col, 'prediction')
# test data w/ QRF median
qrf_median_metrics = compute_metrics_by_fold(test_df_qrf, target_col, "q50")
# test data w/ QRF 10p and 90p
interval_metrics = compute_interval_metrics_by_fold(test_df_qrf, target_col, 'q10', 'q90')

## ABSOLUTE INTENSITIES
# test data
convt_log_metrics = compute_metrics_by_fold(test_df, 'log_region_capital_intensity_USD_per_million_tonne', 'predicted_log_capital_intensity_USD_per_million_tonne')
# test data w/ QRF 10p and 90p
interval_metrics = compute_interval_metrics_by_fold(test_df_qrf, 'log_region_capital_intensity_USD_per_million_tonne', 'q10_predicted_log_capital_intensity_USD_per_million_tonne', 'q90_predicted_log_capital_intensity_USD_per_million_tonne')

## COUNTRY AVEARGE NULL MODEL
country_log_metrics = compute_metrics_by_fold(test_df, 'log_region_capital_intensity_USD_per_million_tonne', 'log_country_capital_intensity_USD_per_million_tonne')

### Save metrics -------------------------------------------------------------------------

# relative log metrics (train + test combined)
train_metrics["split"] = "train"
test_metrics["split"]  = "test"
relative_metrics = pd.concat([train_metrics, test_metrics], ignore_index=True)
relative_metrics.to_csv(out_dir / "relative_log_metrics.csv", index=False)

# actual intensities, log scale (test only)
convt_log_metrics["split"] = "test"
convt_log_metrics.to_csv(out_dir / "actual_log_metrics.csv", index=False)

# country average baseline, log scale (test only)
country_log_metrics["split"] = "test"
country_log_metrics.to_csv(out_dir / "country_avg_log_metrics.csv", index=False)

# QRF interval metrics (coverage / width, test only)
interval_metrics["split"] = "test"
interval_metrics.to_csv(out_dir_qrf / "qrf_interval_metrics.csv", index=False)

# QRF median (q50) point-prediction metrics, relative log scale (test only)
qrf_median_metrics["split"] = "test"
qrf_median_metrics.to_csv(out_dir_qrf / "qrf_median_metrics.csv", index=False)

/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/carinamanitius/Library/Python/3.9/lib/python/site-packages/numpy/lib/_function_base_impl.py:2

In [6]:
##### LABOR MODEL METRICS

### Prep data ---------------------------------------------------------------------------

# set target column
target_col = "rtv_log_labor_intensity_jobs_per_million_tonne" 

# set output directory
out_dir = Path(f'{cd}/Results/RF_models/labor_rf_spatial_CV')
out_dir_qrf = Path(f'{cd}/Results/RF_models/labor_qrf_spatial_CV')

# split into test and train
test_df  = labor_rf[labor_rf["split"] == "test"]
train_df = labor_rf[labor_rf["split"] == "train"]
test_df_qrf = labor_qrf[labor_qrf["split"] == "test"]

### Calculate metrics ----------------------------------------------------------------------

## RELATIVE INTENSITIES
# training data
train_metrics = compute_metrics_by_fold(train_df, target_col, 'prediction')
# test data
test_metrics = compute_metrics_by_fold(test_df, target_col, 'prediction')
# test data w/ QRF median
qrf_median_metrics = compute_metrics_by_fold(test_df_qrf, target_col, "q50")
# test data w/ QRF 10p and 90p
interval_metrics = compute_interval_metrics_by_fold(test_df_qrf, target_col, 'q10', 'q90')

## ABSOLUTE INTENSITIES 
# test data
convt_log_metrics = compute_metrics_by_fold(test_df, 'log_region_labor_intensity_jobs_per_million_tonne', 'predicted_log_labor_intensity_jobs_per_million_tonne')
# test data w/ QRF 10p and 90p
interval_metrics = compute_interval_metrics_by_fold(test_df_qrf, 'log_region_labor_intensity_jobs_per_million_tonne', 'q10_predicted_log_labor_intensity_jobs_per_million_tonne', 'q90_predicted_log_labor_intensity_jobs_per_million_tonne')

## COUNTRY AVERAGE NULL MODEL 
country_log_metrics = compute_metrics_by_fold(test_df, 'log_region_labor_intensity_jobs_per_million_tonne', 'log_country_labor_intensity_jobs_per_million_tonne')

### Save metrics -------------------------------------------------------------------------

# relative log metrics (train + test combined)
train_metrics["split"] = "train"
test_metrics["split"]  = "test"
relative_metrics = pd.concat([train_metrics, test_metrics], ignore_index=True)
relative_metrics.to_csv(out_dir / "relative_log_metrics.csv", index=False)

# actual intensities, log scale (test only)
convt_log_metrics["split"] = "test"
convt_log_metrics.to_csv(out_dir / "actual_log_metrics.csv", index=False)

# country average baseline, log scale (test only)
country_log_metrics["split"] = "test"
country_log_metrics.to_csv(out_dir / "country_avg_log_metrics.csv", index=False)

# QRF interval metrics (coverage / width, test only)
interval_metrics["split"] = "test"
interval_metrics.to_csv(out_dir_qrf / "qrf_interval_metrics.csv", index=False)

# QRF median (q50) point-prediction metrics, relative log scale (test only)
qrf_median_metrics["split"] = "test"
qrf_median_metrics.to_csv(out_dir_qrf / "qrf_median_metrics.csv", index=False)
